# ME-263  Lecture 19  

Last class discussed how to evaluate residual $R$ as directional derivative of energy functional $\Pi$ at $u$ in the direction of $v$ and how to evaluate tangent stiffnes $K$ as the directional derivative of residual $R$ at $v$ in the direction of $v$. 

## 1. Residual and tangent stiffness

$$D\Pi(u)[v] = R(u, v)$$
$$DR(u; v)[w] = K(u; v, w)$$

from a first-order Taylor expansion, $f(x+\Delta x) = f(x) + f'(x)\,\Delta x$.   
Applied to the residual:

$$R(u + \Delta u) = R(u) + \nabla_u R(u)\cdot \Delta u = 0$$
$R$ is linear in $\Delta u$, not in $u$.

## 2. Continuous vs. discrete

| | **Continuous** | **Discrete** |
|---|---|---|
| Problem | Find $u$ such that $\Pi(u)$ is stationary | Find $\underline{u}$ such that $\Pi[u_a N_a]$ is stationary |
| Weak form | $R(u;v) = D\Pi(u)[v] = 0\ \ \forall v$ | $R(\underline{u};\phi_j) = 0\ \ \forall j$ |
| Tangent | $DR(u;v)[w] = K(u;v,w)$ | $DR(\underline{u};\phi_i)[\phi_j] = K_{ij}[\underline{u}]$ |  
| Newton-Raphson update equation| $R(u+\delta u;v) = R(u;v) + DR(u;v)[\delta u]$ | $R(\underline{u}) + K(\underline{u})\Delta u \approx R(\underline(u) + \delta \underline u)$|

**Discretisation:** $u_h = u_a N_a$, test function $v = v_j N_j$, trial increment $w = w_j N_j$.

- $R(u,v)$ is **linear in $v$**, so $R(u,v) = R(u,v_j\phi_j)$.
- Equivalent statement: $R_j(\underline{u}) = 0$ where $R_j(\underline{u}) \equiv R(u,\phi_j)$.

**Tangent stiffness:**
$$DR(u;v)[w] = w_j v_i\, DR(u;\phi_i)[\phi_j] = w_j v_i\, K_{ij}(\underline{u})$$


## 3. Coding part 

Key points:
- In the Jacobian, $v$ plays no role (it is only the test function of the residual).
- `uh` is not a trial/test function.
- `uh` is not an argument of the Jacobian form.
- Everything that changes inside the loop is the residual vector and `uh`.

Setup (assumes `mesh`, `V`, `uh`, `bc`, `Res`, `Jac` are already defined in the problem script):

In [29]:
from mpi4py import MPI
from dolfinx import fem,mesh,io
from dolfinx.fem.petsc import LinearProblem
from petsc4py import PETSc
import ufl
import numpy as np

In [30]:
# Slender cantilever: long and thin so it bends a lot under an end load
L, H = 8.0, 1.0
nx, ny = 48, 6
msh = mesh.create_rectangle(MPI.COMM_WORLD, [[0.0, 0.0], [L, H]], [nx, ny], mesh.CellType.triangle)
tdim = msh.topology.dim
fdim = tdim - 1

In [31]:
V = fem.functionspace(msh, ("Lagrange",1,(2,)))
v = ufl.TestFunction(V)
du = ufl.TrialFunction(V)

# plane strain neo_Hookean moduli
E, nu = 1.0, 0.3
mu = E/(2*(1+nu))
lmbda = E*nu/((1+nu)*(1-2*nu))

In [32]:
# clamp the LEFT edge: u = 0 (homogeneous, so u=0 satisfies it and every increment is 0 there)
left = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0],0.0))
dofsL = fem.locate_dofs_topological(V, fdim, left)
bc = fem.dirichletbc(fem.Function(V), dofsL)

In [33]:
# traction load on the RIGHT edge (x = L), a downward shear, scaled by a load factor lam
right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], L))
mt = mesh.meshtags(msh, fdim, np.sort(right), np.full(len(right), 1, dtype=np.int32))  # [VERIFY 0.11]
ds = ufl.Measure("ds", domain=msh, subdomain_data=mt)   

tmax = fem.Constant(msh, np.array([0.0, -0.002]))       # TUNE: traction magnitude on the right edge      
lam =  fem.Constant(msh, PETSc.ScalarType(1.0))         # load factor - lam=1 is the "working" load

In [34]:
uh = fem.Function(V)                          # the evolving state, shared by all schemes
F = ufl.Identity(2) + ufl.grad(uh)
Jd = ufl.det(F)                          
Ic = ufl.tr(F.T*F) + 1                        # plane strain: +1 (z-stretch)
W = (mu/2)*(Ic-3) - mu*ufl.ln(Jd) + (lmbda/2)*ufl.ln(Jd)** 2
Pi = W*ufl.dx - lam*ufl.inner(tmax, uh)*ds(1) # total potential (internal energy - traction work)
 
Res = ufl.derivative(Pi, uh, v)               # R(u; v) -- driven to zero
Jac = ufl.derivative(Res, uh, du)             # K(u; v, du) = consistent tangent
Res_form = fem.form(Res)

In [35]:
lam.value = 1.0                      # scale factor
uh.x.array[:] = 0

problem = LinearProblem(Jac, -Res, bcs=[bc],
                        petsc_options_prefix="full_",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})

max_iter = 40
for k in range(max_iter):
    r = fem.petsc.assemble_vector(Res_form)
    fem.petsc.set_bc(r, [bc], alpha=0.0)
    rnorm = r.norm()
    print(f"iter {k:2d}: |R| = {rnorm:.3e}")

    if rnorm < 1e-8:
        print(f"tip disp = {uh.x.array[1::2].min():.4f}")
        with io.XDMFFile(msh.comm, "hand_newton.xdmf", "w") as file:
            file.write_mesh(msh)
            file.write_function(uh)
        break
    else:
        delta = problem.solve()
        uh.x.array[:] += delta.x.array
u_full = uh.x.array.copy()

iter  0: |R| = 7.817e-04
iter  1: |R| = 3.392e-01
iter  2: |R| = 7.049e-02
iter  3: |R| = 2.369e-03
iter  4: |R| = 8.482e-04
iter  5: |R| = 1.709e-05
iter  6: |R| = 4.992e-07
iter  7: |R| = 1.218e-11
tip disp = -3.0175


In [36]:
lam.value = 3.0
uh.x.array[:] = 0

problem = LinearProblem(Jac, -Res, bcs=[bc], petsc_options_prefix="full_",
                        petsc_options={"ksp_type":"preonly", "pc_type": "lu"})

max_iter = 40
for k in range(max_iter):
    r = fem.petsc.assemble_vector(Res_form)
    fem.petsc.set_bc(r, [bc], alpha=0.0)
    rnorm = r.norm()
    print(f"iter {k:2d}: |R| = {rnorm:.3e}")
    if rnorm<1e-8:
        #print(f"tip disp = {uh[1 :: 2].min():+.4f}")
        uh.name = "u"
        with io.XDMFFile(msh.comm, "large_newton.xdmf", "w") as file:
            file.write_mesh(msh)
            file.write_function(uh)
        break
    else:
        delta = problem.solve()
        uh.x.array[:] += delta.x.array

iter  0: |R| = 2.345e-03
iter  1: |R| = 1.332e+00
iter  2: |R| = 1.471e+01
iter  3: |R| = nan
iter  4: |R| = nan
iter  5: |R| = nan
iter  6: |R| = nan
iter  7: |R| = nan
iter  8: |R| = nan
iter  9: |R| = nan
iter 10: |R| = nan
iter 11: |R| = nan
iter 12: |R| = nan
iter 13: |R| = nan
iter 14: |R| = nan
iter 15: |R| = nan
iter 16: |R| = nan
iter 17: |R| = nan
iter 18: |R| = nan
iter 19: |R| = nan
iter 20: |R| = nan
iter 21: |R| = nan
iter 22: |R| = nan
iter 23: |R| = nan
iter 24: |R| = nan
iter 25: |R| = nan
iter 26: |R| = nan
iter 27: |R| = nan
iter 28: |R| = nan
iter 29: |R| = nan
iter 30: |R| = nan
iter 31: |R| = nan
iter 32: |R| = nan
iter 33: |R| = nan
iter 34: |R| = nan
iter 35: |R| = nan
iter 36: |R| = nan
iter 37: |R| = nan
iter 38: |R| = nan
iter 39: |R| = nan


### Large loads
For large loads the plain code above does not work (Newton diverges from $u=0$), so we provide small step increments — i.e. *force control*: ramp the load $\lambda$ from 0 to its final value and solve a Newton problem at every step, starting from the previous converged solution.


In [37]:
#LAM_LARGE = 3.0
uh.x.array[:] = 0
problem = LinearProblem(Jac, -Res, bcs=[bc], petsc_options_prefix="large_load_",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})

max_iter = 40
for lstep in range(0,5):
    lam.value = float(lstep)*3.0/4.0
    for k in range(max_iter):
        r = fem.petsc.assemble_vector(Res_form)
        fem.petsc.set_bc(r, bcs=[bc], alpha=0.0)
        rnorm = r.norm()
        print(f"iter {k:2d}: |R| = {rnorm:.3e}")
        if rnorm<1e-8:
            #print(f"tip disp = {uh[1 :: 2].min():+.4f}")
            uh.name = "u"
            with io.XDMFFile(msh.comm, f"large_load_{{lstep}}.xdmf", "w") as file:
                file.write_mesh(msh)
                file.write_function(uh)
            break
    else:
        delta = problem.solve()
        uh.x.array[:] += delta.x.array


iter  0: |R| = 0.000e+00
iter  0: |R| = 5.863e-04
iter  1: |R| = 5.863e-04
iter  2: |R| = 5.863e-04
iter  3: |R| = 5.863e-04
iter  4: |R| = 5.863e-04
iter  5: |R| = 5.863e-04
iter  6: |R| = 5.863e-04
iter  7: |R| = 5.863e-04
iter  8: |R| = 5.863e-04
iter  9: |R| = 5.863e-04
iter 10: |R| = 5.863e-04
iter 11: |R| = 5.863e-04
iter 12: |R| = 5.863e-04
iter 13: |R| = 5.863e-04
iter 14: |R| = 5.863e-04
iter 15: |R| = 5.863e-04
iter 16: |R| = 5.863e-04
iter 17: |R| = 5.863e-04
iter 18: |R| = 5.863e-04
iter 19: |R| = 5.863e-04
iter 20: |R| = 5.863e-04
iter 21: |R| = 5.863e-04
iter 22: |R| = 5.863e-04
iter 23: |R| = 5.863e-04
iter 24: |R| = 5.863e-04
iter 25: |R| = 5.863e-04
iter 26: |R| = 5.863e-04
iter 27: |R| = 5.863e-04
iter 28: |R| = 5.863e-04
iter 29: |R| = 5.863e-04
iter 30: |R| = 5.863e-04
iter 31: |R| = 5.863e-04
iter 32: |R| = 5.863e-04
iter 33: |R| = 5.863e-04
iter 34: |R| = 5.863e-04
iter 35: |R| = 5.863e-04
iter 36: |R| = 5.863e-04
iter 37: |R| = 5.863e-04
iter 38: |R| = 5.863e-04


### Modified Newton method ("cheat way")

If the residual has to go to zero anyway, then we don't need to update the stiffness matrix every iteration — instead update it once every 4 iterations.

- If the iterations converge, the solution is equally correct (the residual is what defines the solution; the tangent only affects the convergence path).
- It works because the change in solution is small, so we are reusing the tangent stiffness from an earlier point on the curve.
- Trade-off: takes more iterations to solve (convergence is no longer quadratic), but each iteration is cheaper (no re-assembly / refactorisation).

**`Jac_fix`** → a new form where there is no `uh`:

- `Jac_fix` → evaluated at `u_fixed`
- `-Res` → evaluated at `uh`  
Eventhough tangent stiffnes $K$ is fixed, the residual $R$ is still updated every iteration.  

In [38]:
lam.value = 0.1
lag = 4
uh.x.array[:] = 0
ufix = fem.Function(V)
Jac_fix = ufl.replace(Jac, {uh:ufix})
problem = LinearProblem(Jac_fix, -Res, bcs=[bc], petsc_options_prefix="lag_",
                        petsc_options={"ksp_type":"preonly", "pc_type":"lu"})

max_iter = 40
for k in range(max_iter):
    r = fem.petsc.assemble_vector(Res_form)
    fem.petsc.set_bc(r, [bc], alpha=0.0)
    rnorm = r.norm()
    print(f"iter {k:2d}: |R| = {rnorm:.3e}")
    if rnorm < 1e-8:
        #print(f"tip disp = {uh[1 :: 2].min():+.4f}")
        uh.name = "u"
        with io.XDMFFile(msh.comm, "lag_newton.xdmf", "w") as file:
            file.write_mesh(msh)
            file.write_function(uh)
        break
    elif k%lag ==0:
        ufix.x.array[:] = uh.x.array
    else:
        delta = problem.solve()
        uh.x.array[:] += delta.x.array

iter  0: |R| = 7.817e-05
iter  1: |R| = 7.817e-05
iter  2: |R| = 4.272e-03
iter  3: |R| = 2.948e-03
iter  4: |R| = 7.473e-04
iter  5: |R| = 7.473e-04
iter  6: |R| = 2.363e-03
iter  7: |R| = 2.054e-03
iter  8: |R| = 1.073e-03
iter  9: |R| = 1.073e-03
iter 10: |R| = 1.186e-06
iter 11: |R| = 4.380e-07
iter 12: |R| = 1.302e-07
iter 13: |R| = 1.302e-07
iter 14: |R| = 5.047e-10


### Splitting the residual: $R_{int}$ and $\lambda R_{ext}$

$$R(u,v) = \underbrace{\int_\Omega P : \delta v \, d\Omega}_{R_{int}} \;+\; \lambda \underbrace{\int_{\Gamma_N} t\cdot v \, ds}_{R_{ext}} \;\to\; 0\quad \forall v$$

where $P$ is the first Piola–Kirchhoff stress ($u$ is in $P$, i.e. $P=P(u)$).

You fix $R_{int}$ (frozen at $u_i$), and update $R_{ext}$ with $\lambda$.

Approximating $K$ → the solid is still in equilibrium:

$$R(u+\Delta u_\lambda)\big|_{\lambda=1.1} = \underbrace{R(u_i)}_{\text{frozen}} + \underbrace{K(u_i)}_{\text{frozen}}\underbrace{\Delta u_\lambda}_ {\text{updates with }\lambda}$$

with the two parts

- $R_{int}(u_i)$ → frozen
- $\lambda R_{ext}$ → updates with $\lambda$

so $R(u_i + \Delta u_\lambda)$ is computed while making a part of the residual fixed.

Because the stiffness matrix and part of the residual are held fixed, equilibrium (full residual = 0) will not be satisfied at the predicted state — it is only an approximation (a predictor), which is why further Newton iterations are still needed.